In [ ]:
import math
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests
import yfinance as yf

## SPX and SPX Option

## SPX Options (Fast Path - Polygon Flat Files)

In [ ]:
# Fast SPX options build: flat-file probe + API fallback
# Delta-only mode + checkpoint/resume

import io
import gzip
import os
import math
import requests
import numpy as np
import pandas as pd
import yfinance as yf
from concurrent.futures import ThreadPoolExecutor, as_completed

start_date = "2024-06-01"
end_date = "2026-05-01"
RISK_FREE_RATE = 0.04
MAX_WORKERS = 24

# Tuning knobs:
# - BUILD_MAX_DAYS=5 for quick smoke test; 0 means full range.
# - MAX_CONTRACTS_PER_DAY=500 for speed; 0 means no cap.
MAX_DAYS = int(os.getenv("BUILD_MAX_DAYS", "0") or "0")
MAX_CONTRACTS_PER_DAY = int(os.getenv("MAX_CONTRACTS_PER_DAY", "0") or "0")
CHECKPOINT_EVERY_DAYS = int(os.getenv("CHECKPOINT_EVERY_DAYS", "1") or "1")

CONTRACTS_URL = "https://api.massive.com/v3/reference/options/contracts"
OUTPUT_CSV = "spx_options.csv"
OUTPUT_PARQUET = "spx_options.parquet"


def _read_api_key():
    key = os.getenv("MASSIVE_API_KEY") or os.getenv("POLYGON_API_KEY") or os.getenv("Polygon_API_Key")
    if key:
        return key.strip().strip('"').strip("'")
    if os.path.exists(".env"):
        with open(".env", "r", encoding="utf-8") as f:
            for line in f:
                if "=" not in line:
                    continue
                k, v = line.split("=", 1)
                if k.strip() in {"MASSIVE_API_KEY", "POLYGON_API_KEY", "Polygon_API_Key"}:
                    return v.strip().strip('"').strip("'")
    return None


API_KEY = _read_api_key()
if not API_KEY:
    raise ValueError("Set MASSIVE_API_KEY (or POLYGON_API_KEY) in env/.env.")

HEADERS = {"Authorization": f"Bearer {API_KEY}"}

start_dt = pd.to_datetime(start_date)
end_dt = pd.to_datetime(end_date)
if start_dt > end_dt:
    raise ValueError("start_date must be <= end_date")


def _flatfile_candidate_urls(day: pd.Timestamp):
    yyyy = day.strftime("%Y")
    mm = day.strftime("%m")
    d_dash = day.strftime("%Y-%m-%d")
    d_compact = day.strftime("%Y%m%d")
    return [
        f"https://files.polygon.io/us_options_opra/day_aggs_v1/{yyyy}/{mm}/{d_dash}.csv.gz",
        f"https://files.polygon.io/us_options_opra/day_aggs_v1/{yyyy}/{mm}/{d_compact}.csv.gz",
        f"https://files.polygon.io/us_options_opra/day_aggs_v1/{d_dash}.csv.gz",
        f"https://files.polygon.io/us_options_opra/day_aggs_v1/{d_compact}.csv.gz",
    ]


def fetch_flatfile_day(day: pd.Timestamp) -> pd.DataFrame:
    for url in _flatfile_candidate_urls(day):
        for req_url, headers in ((url, HEADERS), (f"{url}?apiKey={API_KEY}", None)):
            try:
                r = requests.get(req_url, headers=headers, timeout=90)
                if r.status_code in (401, 403, 404):
                    continue
                r.raise_for_status()
                content = r.content
                if req_url.endswith(".gz") or content[:2] == b"\x1f\x8b":
                    with gzip.GzipFile(fileobj=io.BytesIO(content), mode="rb") as gz:
                        raw = gz.read()
                    return pd.read_csv(io.BytesIO(raw))
                return pd.read_csv(io.BytesIO(content))
            except Exception:
                continue
    return pd.DataFrame()


def fetch_contracts_for_as_of(as_of_date: str) -> list[dict]:
    as_of_ts = pd.to_datetime(as_of_date)
    exp_lt = (as_of_ts + pd.Timedelta(days=183)).strftime("%Y-%m-%d")
    params = {
        "underlying_ticker": "SPX",
        "as_of": as_of_date,
        "expiration_date.gte": as_of_date,
        "expiration_date.lt": exp_lt,
        "limit": 1000,
        "sort": "ticker",
        "order": "asc",
    }

    rows = []
    next_url = CONTRACTS_URL
    while next_url:
        response = requests.get(next_url, headers=HEADERS, params=params, timeout=60)
        response.raise_for_status()
        payload = response.json()
        page_rows = payload.get("results", [])
        for r in page_rows:
            r["as_of_date"] = as_of_date
        rows.extend(page_rows)
        next_url = payload.get("next_url")
        params = None
    return rows


def fetch_daily_close_for_ticker(ticker: str, as_of_date: str) -> dict:
    url = f"https://api.massive.com/v2/aggs/ticker/{ticker}/range/1/day/{as_of_date}/{as_of_date}"
    params = {"adjusted": True, "limit": 5}
    try:
        response = requests.get(url, headers=HEADERS, params=params, timeout=30)
        response.raise_for_status()
        payload = response.json()
        results = payload.get("results", [])
        if results:
            bar = results[0]
            return {
                "close": bar.get("c"),
                "open": bar.get("o"),
                "high": bar.get("h"),
                "low": bar.get("l"),
                "volume": bar.get("v"),
            }
    except Exception:
        return {"close": None, "open": None, "high": None, "low": None, "volume": None}
    return {"close": None, "open": None, "high": None, "low": None, "volume": None}


def enrich_rows_with_daily_close(rows: list[dict], as_of_date: str, max_workers: int = MAX_WORKERS) -> list[dict]:
    if not rows:
        return rows

    ticker_to_indices = {}
    for idx, row in enumerate(rows):
        ticker = row.get("ticker")
        if ticker:
            ticker_to_indices.setdefault(ticker, []).append(idx)

    metrics_map = {}
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        future_map = {
            ex.submit(fetch_daily_close_for_ticker, ticker, as_of_date): ticker
            for ticker in ticker_to_indices
        }
        for future in as_completed(future_map):
            ticker = future_map[future]
            try:
                metrics_map[ticker] = future.result()
            except Exception:
                metrics_map[ticker] = {"close": None, "open": None, "high": None, "low": None, "volume": None}

    for ticker, indices in ticker_to_indices.items():
        m = metrics_map.get(ticker, {})
        for idx in indices:
            rows[idx]["close"] = m.get("close")
            rows[idx]["open"] = m.get("open")
            rows[idx]["high"] = m.get("high")
            rows[idx]["low"] = m.get("low")
            rows[idx]["volume"] = m.get("volume")

    return rows


def norm_cdf(x: float) -> float:
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def bs_price(S: float, K: float, T: float, r: float, sigma: float, option_type: str) -> float:
    sqrtT = math.sqrt(T)
    d1 = (math.log(S / K) + (r + 0.5 * sigma * sigma) * T) / (sigma * sqrtT)
    d2 = d1 - sigma * sqrtT
    if option_type.lower() == "call":
        return S * norm_cdf(d1) - K * math.exp(-r * T) * norm_cdf(d2)
    return K * math.exp(-r * T) * norm_cdf(-d2) - S * norm_cdf(-d1)


def bs_delta(S: float, K: float, T: float, r: float, sigma: float, option_type: str) -> float:
    sqrtT = math.sqrt(T)
    d1 = (math.log(S / K) + (r + 0.5 * sigma * sigma) * T) / (sigma * sqrtT)
    if option_type.lower() == "call":
        return norm_cdf(d1)
    return norm_cdf(d1) - 1.0


def implied_vol_and_delta(price_mkt: float, S: float, K: float, T: float, r: float, option_type: str):
    if price_mkt is None or S is None or K is None or T is None:
        return None
    if (not np.isfinite(price_mkt)) or (not np.isfinite(S)) or (not np.isfinite(K)) or (not np.isfinite(T)):
        return None
    if S <= 0 or K <= 0 or T <= 0 or price_mkt <= 0:
        return None

    low, high = 1e-4, 5.0
    for _ in range(60):
        mid = 0.5 * (low + high)
        pmid = bs_price(S, K, T, r, mid, option_type)
        if abs(pmid - price_mkt) < 1e-4:
            return bs_delta(S, K, T, r, mid, option_type)
        if pmid > price_mkt:
            high = mid
        else:
            low = mid

    sigma = 0.5 * (low + high)
    return bs_delta(S, K, T, r, sigma, option_type)


def parse_polygon_option_ticker(ticker: str):
    t = str(ticker)
    if t.startswith("O:"):
        t = t[2:]
    m = re.match(r"^(?P<root>[A-Z]+?)(?P<yymmdd>\d{6})(?P<cp>[CP])(?P<strike>\d{8})$", t)
    if not m:
        return None
    exp = pd.to_datetime(m.group("yymmdd"), format="%y%m%d", errors="coerce")
    if pd.isna(exp):
        return None
    root = m.group("root")
    option_type = "call" if m.group("cp") == "C" else "put"
    strike = int(m.group("strike")) / 1000.0
    return root, exp, option_type, strike


def _load_existing_checkpoint():
    if os.path.exists(OUTPUT_PARQUET):
        try:
            df = pd.read_parquet(OUTPUT_PARQUET)
            return df
        except Exception:
            pass
    if os.path.exists(OUTPUT_CSV):
        try:
            return pd.read_csv(OUTPUT_CSV, parse_dates=["as_of_date", "expiration_date"])
        except Exception:
            pass
    return pd.DataFrame()


spx_underlying = yf.download(
    "^SPX",
    start=start_dt.strftime("%Y-%m-%d"),
    end=(end_dt + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
    progress=False,
    auto_adjust=False,
)
if isinstance(spx_underlying.columns, pd.MultiIndex):
    spx_underlying = spx_underlying.copy()
    spx_underlying.columns = spx_underlying.columns.droplevel(1)
spx_underlying = spx_underlying.reset_index()
spx_underlying["Date"] = pd.to_datetime(spx_underlying["Date"]).dt.normalize()
spx_close_map = dict(zip(spx_underlying["Date"], spx_underlying["Close"]))

existing_df = _load_existing_checkpoint()
processed_dates = set()
if not existing_df.empty and "as_of_date" in existing_df.columns:
    processed_dates = set(pd.to_datetime(existing_df["as_of_date"], errors="coerce").dt.strftime("%Y-%m-%d").dropna().unique())

business_days = pd.bdate_range(start_dt, end_dt)
if MAX_DAYS > 0:
    business_days = business_days[:MAX_DAYS]

if len(business_days) == 0:
    raise ValueError("No business days in selected range.")

print(f"Processing {len(business_days)} business days from {business_days[0].date()} to {business_days[-1].date()}\n")
print(f"Checkpoint loaded rows: {len(existing_df):,}; processed days: {len(processed_dates):,}")

flat_enabled = False
for d in business_days[:2]:
    probe = fetch_flatfile_day(d)
    if not probe.empty:
        flat_enabled = True
        break
print("Flat-file mode:", "ENABLED" if flat_enabled else "DISABLED (fallback to API mode)")

master_df = existing_df.copy()
new_days = 0

for i, day in enumerate(business_days, start=1):
    as_of = day.strftime("%Y-%m-%d")
    if as_of in processed_dates:
        print(f"[{i}/{len(business_days)}] Skip {as_of} (checkpoint)")
        continue

    print(f"[{i}/{len(business_days)}] Load {as_of}")
    day_df = pd.DataFrame()

    try:
        if flat_enabled:
            ff = fetch_flatfile_day(day)
            if not ff.empty and "ticker" in ff.columns:
                ff = ff.rename(columns={"o": "open", "h": "high", "l": "low", "c": "close", "v": "volume"})
                parsed = ff["ticker"].apply(parse_polygon_option_ticker)
                parsed_df = parsed.apply(
                    lambda x: pd.Series(x, index=["root", "expiration_date", "contract_type", "strike_price"])
                    if x is not None
                    else pd.Series([None, pd.NaT, None, np.nan], index=["root", "expiration_date", "contract_type", "strike_price"])
                )
                ff = pd.concat([ff, parsed_df], axis=1)
                ff = ff[ff["root"].isin(["SPX", "SPXW"])].copy()
                ff["as_of_date"] = day.normalize()
                ff["SPX_close"] = spx_close_map.get(day.normalize(), np.nan)
                keep_cols = [
                    "as_of_date", "ticker", "contract_type", "strike_price", "expiration_date",
                    "SPX_close", "open", "high", "low", "close", "volume"
                ]
                keep_cols = [c for c in keep_cols if c in ff.columns]
                day_df = ff[keep_cols].copy()

        if day_df.empty:
            rows = fetch_contracts_for_as_of(as_of)
            if MAX_CONTRACTS_PER_DAY > 0:
                rows = rows[:MAX_CONTRACTS_PER_DAY]
            rows = enrich_rows_with_daily_close(rows, as_of, max_workers=MAX_WORKERS)
            spx_close = spx_close_map.get(day.normalize(), np.nan)
            for r in rows:
                r["SPX_close"] = spx_close
            api_df = pd.DataFrame(rows)
            if not api_df.empty:
                keep_cols = [
                    "as_of_date", "ticker", "contract_type", "strike_price", "expiration_date",
                    "SPX_close", "open", "high", "low", "close", "volume"
                ]
                keep_cols = [c for c in keep_cols if c in api_df.columns]
                day_df = api_df[keep_cols].copy()

        if day_df.empty:
            print("  -> no rows")
            continue

        day_df = day_df.copy()
        exp_ts = pd.to_datetime(day_df["expiration_date"], errors="coerce")
        asof_ts = pd.to_datetime(day_df["as_of_date"], errors="coerce")
        day_df.loc[:, "expiration_date"] = exp_ts
        day_df.loc[:, "as_of_date"] = asof_ts

        # Avoid .dt accessor failures when a provider response carries mixed/object datetime values.
        dte_days = (exp_ts - asof_ts) / np.timedelta64(1, "D")
        day_df.loc[:, "days_to_strike"] = pd.to_numeric(dte_days, errors="coerce")

        day_df = day_df[(day_df["days_to_strike"] > 0) & (day_df["days_to_strike"] < 183)].copy()
        if day_df.empty:
            print("  -> all rows filtered by DTE")
            continue

        day_df.loc[:, "T"] = day_df["days_to_strike"] / 252.0
        day_df.loc[:, "delta"] = np.nan
        for idx, row in day_df.iterrows():
            d = implied_vol_and_delta(
                row.get("close"),
                row.get("SPX_close"),
                row.get("strike_price"),
                row.get("T"),
                RISK_FREE_RATE,
                row.get("contract_type"),
            )
            if d is not None and np.isfinite(d):
                day_df.at[idx, "delta"] = float(d)

        day_df = day_df.drop(columns=["T"], errors="ignore")
        final_cols = [
            "as_of_date", "ticker", "contract_type", "strike_price", "expiration_date", "days_to_strike",
            "SPX_close", "close", "delta", "open", "high", "low", "volume"
        ]
        final_cols = [c for c in final_cols if c in day_df.columns]
        day_df = day_df[final_cols]

        master_df = pd.concat([master_df, day_df], ignore_index=True)
        master_df = master_df.drop_duplicates(subset=["ticker", "as_of_date"])
        processed_dates.add(as_of)
        new_days += 1

        if new_days % CHECKPOINT_EVERY_DAYS == 0:
            master_df.to_csv(OUTPUT_CSV, index=False)
            master_df.to_parquet(OUTPUT_PARQUET, index=False)
            print(f"  -> checkpoint saved; total rows: {len(master_df):,}")
        else:
            print(f"  -> rows kept: {len(day_df):,}; total rows: {len(master_df):,}")

    except Exception as exc:
        print(f"  !! Error for {as_of}: {exc}")

master_df.to_csv(OUTPUT_CSV, index=False)
master_df.to_parquet(OUTPUT_PARQUET, index=False)
print(f"\nDone! Saved {len(master_df):,} rows")
print(f"CSV: {OUTPUT_CSV}")
print(f"Parquet: {OUTPUT_PARQUET}")
master_df.head()




  -> checkpoint saved; total rows: 6,935,958
[371/500] Load 2025-11-03
  -> checkpoint saved; total rows: 6,956,714
[372/500] Load 2025-11-04
  -> checkpoint saved; total rows: 6,977,028
[373/500] Load 2025-11-05
  -> checkpoint saved; total rows: 6,997,248
[374/500] Load 2025-11-06
  -> checkpoint saved; total rows: 7,017,648
[375/500] Load 2025-11-07
  -> checkpoint saved; total rows: 7,037,798
[376/500] Load 2025-11-10
  -> checkpoint saved; total rows: 7,058,386
[377/500] Load 2025-11-11
  -> checkpoint saved; total rows: 7,078,526
[378/500] Load 2025-11-12
  -> checkpoint saved; total rows: 7,098,640
[379/500] Load 2025-11-13
  -> checkpoint saved; total rows: 7,118,734
[380/500] Load 2025-11-14
  -> checkpoint saved; total rows: 7,139,014
[381/500] Load 2025-11-17
  -> checkpoint saved; total rows: 7,159,870
[382/500] Load 2025-11-18
  -> checkpoint saved; total rows: 7,180,276
[383/500] Load 2025-11-19
  -> checkpoint saved; total rows: 7,200,946
[384/500] Load 2025-11-20
  -> c

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4012\1494806268.py:379: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  master_df = pd.concat([master_df, day_df], ignore_index=True)


  -> checkpoint saved; total rows: 7,325,752
[390/500] Load 2025-11-28
  -> checkpoint saved; total rows: 7,346,446
[391/500] Load 2025-12-01
  -> checkpoint saved; total rows: 7,367,338
[392/500] Load 2025-12-02
  -> checkpoint saved; total rows: 7,388,038
[393/500] Load 2025-12-03
  -> checkpoint saved; total rows: 7,408,638
[394/500] Load 2025-12-04
  -> checkpoint saved; total rows: 7,429,132
[395/500] Load 2025-12-05
  -> checkpoint saved; total rows: 7,449,248
[396/500] Load 2025-12-08
  !! Error for 2025-12-08: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
[397/500] Load 2025-12-09
  !! Error for 2025-12-09: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
[398/500] Load 2025-12-10
  !! Error for 2025-12-10: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
[399/500] Load 2025-12-11
  -> checkpoint saved; total rows: 7,469,678
[400/500] Load 2025-

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4012\1494806268.py:379: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  master_df = pd.concat([master_df, day_df], ignore_index=True)


  -> checkpoint saved; total rows: 7,669,436
[410/500] Load 2025-12-26
  -> checkpoint saved; total rows: 7,689,730
[411/500] Load 2025-12-29
  -> checkpoint saved; total rows: 7,709,900
[412/500] Load 2025-12-30
  -> checkpoint saved; total rows: 7,730,524
[413/500] Load 2025-12-31
  -> checkpoint saved; total rows: 7,750,330
[414/500] Load 2026-01-01


...

  -> checkpoint saved; total rows: 7,770,136
[415/500] Load 2026-01-02
  -> checkpoint saved; total rows: 7,790,246
[416/500] Load 2026-01-05
  -> checkpoint saved; total rows: 7,810,296
[417/500] Load 2026-01-06
  -> checkpoint saved; total rows: 7,830,398
[418/500] Load 2026-01-07
  -> checkpoint saved; total rows: 7,850,846
[419/500] Load 2026-01-08
  -> checkpoint saved; total rows: 7,871,182
[420/500] Load 2026-01-09
  -> checkpoint saved; total rows: 7,891,270
[421/500] Load 2026-01-12
  -> checkpoint saved; total rows: 7,911,536
[422/500] Load 2026-01-13
  -> checkpoint saved; total rows: 7,931,774
[423/500] Load 2026-01-14
  -> checkpoint saved; total rows: 7,951,712
[424/500] Load 2026-01-15
  -> checkpoint saved; total rows: 7,971,956
[425/500] Load 2026-01-16
  -> checkpoint saved; total rows: 7,991,098
[426/500] Load 2026-01-19


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4012\1494806268.py:379: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  master_df = pd.concat([master_df, day_df], ignore_index=True)


  -> checkpoint saved; total rows: 8,010,240
[427/500] Load 2026-01-20
  -> checkpoint saved; total rows: 8,030,432
[428/500] Load 2026-01-21
  -> checkpoint saved; total rows: 8,051,792
[429/500] Load 2026-01-22
  -> checkpoint saved; total rows: 8,073,044
[430/500] Load 2026-01-23
  -> checkpoint saved; total rows: 8,094,060
[431/500] Load 2026-01-26
  -> checkpoint saved; total rows: 8,114,896
[432/500] Load 2026-01-27
  -> checkpoint saved; total rows: 8,135,802
[433/500] Load 2026-01-28
  -> checkpoint saved; total rows: 8,156,552
[434/500] Load 2026-01-29
  -> checkpoint saved; total rows: 8,177,222
[435/500] Load 2026-01-30
  -> checkpoint saved; total rows: 8,197,372
[436/500] Load 2026-02-02
  -> checkpoint saved; total rows: 8,217,624
[437/500] Load 2026-02-03
  -> checkpoint saved; total rows: 8,237,806
[438/500] Load 2026-02-04
  -> checkpoint saved; total rows: 8,258,062
[439/500] Load 2026-02-05
  -> checkpoint saved; total rows: 8,278,398
[440/500] Load 2026-02-06
  -> c

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4012\1494806268.py:379: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  master_df = pd.concat([master_df, day_df], ignore_index=True)


  -> checkpoint saved; total rows: 8,421,074
[447/500] Load 2026-02-17
  -> checkpoint saved; total rows: 8,441,214
[448/500] Load 2026-02-18
  -> checkpoint saved; total rows: 8,461,360
[449/500] Load 2026-02-19
  -> checkpoint saved; total rows: 8,481,694
[450/500] Load 2026-02-20
  -> checkpoint saved; total rows: 8,501,682
[451/500] Load 2026-02-23
  -> checkpoint saved; total rows: 8,522,016
[452/500] Load 2026-02-24
  -> checkpoint saved; total rows: 8,542,376
[453/500] Load 2026-02-25
  -> checkpoint saved; total rows: 8,562,626
[454/500] Load 2026-02-26
  -> checkpoint saved; total rows: 8,583,072
[455/500] Load 2026-02-27
  -> checkpoint saved; total rows: 8,602,900
[456/500] Load 2026-03-02
  -> checkpoint saved; total rows: 8,622,884
[457/500] Load 2026-03-03
  -> checkpoint saved; total rows: 8,642,730
[458/500] Load 2026-03-04
  -> checkpoint saved; total rows: 8,662,838
[459/500] Load 2026-03-05
  -> checkpoint saved; total rows: 8,682,822
[460/500] Load 2026-03-06
  -> c

## SPX


In [ ]:
spx_start_date = "2024-01-01"
spx_end_date = "2026-05-01"

# yfinance end date is exclusive, so add one day to include spx_end_date
spx_end_exclusive = (pd.Timestamp(spx_end_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

spx_data = yf.download(
    "^SPX",
    start=spx_start_date,
    end=spx_end_exclusive,
    progress=False,
    auto_adjust=False,
)

# Single-ticker downloads use a MultiIndex; drop ticker level for a flat CSV
if isinstance(spx_data.columns, pd.MultiIndex):
    spx_data = spx_data.copy()
    spx_data.columns = spx_data.columns.droplevel(1)

spx_data = spx_data.reset_index()
log_ret = np.log(spx_data["Close"] / spx_data["Close"].shift(1))
spx_data["RV22"] = np.sqrt((252.0 / 22.0) * log_ret.pow(2).rolling(22, min_periods=22).sum())
spx_data.to_csv("spx_data.csv", index=False)

print(f"Downloaded {len(spx_data)} SPX (^SPX) rows from {spx_start_date} to {spx_end_date}")
spx_data.head()

## VVIX

In [ ]:
vvix_start_date = "2024-01-01"
vvix_end_date = "2026-05-01"

# yfinance end date is exclusive
vvix_end_exclusive = (pd.Timestamp(vvix_end_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

# CBOE VVIX (volatility of VIX); Yahoo ticker ^VVIX
time.sleep(1)  # small gap helps avoid yfinance rate limits after other download cells

vvix_data = yf.download(
    "^VVIX",
    start=vvix_start_date,
    end=vvix_end_exclusive,
    progress=False,
    auto_adjust=False,
)

if isinstance(vvix_data.columns, pd.MultiIndex):
    vvix_data = vvix_data.copy()
    vvix_data.columns = vvix_data.columns.droplevel(1)

vvix_data = vvix_data.reset_index()
vvix_data.to_csv("vvix_data.csv", index=False)

print(f"Downloaded {len(vvix_data)} VVIX (^VVIX) rows from {vvix_start_date} to {vvix_end_date}")
vvix_data.head()

## VIX

In [ ]:
vix_start_date = "2024-01-01"
vix_end_date = "2026-05-01"

# yfinance end date is exclusive
vix_end_exclusive = (pd.Timestamp(vix_end_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

# CBOE VIX (volatility of VIX); Yahoo ticker ^VIX
time.sleep(1)  # small gap helps avoid yfinance rate limits after other download cells

vix_data = yf.download(
    "^VIX",
    start=vix_start_date,
    end=vix_end_exclusive,
    progress=False,
    auto_adjust=False,
)

if isinstance(vix_data.columns, pd.MultiIndex):
    vix_data = vix_data.copy()
    vix_data.columns = vix_data.columns.droplevel(1)

vix_data = vix_data.reset_index()
vix_data.to_csv("vix_data.csv", index=False)

print(f"Downloaded {len(vix_data)} VIX (^VIX) rows from {vix_start_date} to {vix_end_date}")
vix_data.head()